<a href="https://colab.research.google.com/github/serahnjogu-new/Climate-and-health-risk-prediction/blob/main/Climate_and_health_risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from google.colab import files

warnings.filterwarnings('ignore')

# 1. LOAD DATA
train   = pd.read_csv('/content/Train.csv')
test    = pd.read_csv('/content/Test.csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission.csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. OPTIMIZED FEATURE ENGINEERING (The 0.838 Formula + Temperature Anomalies)
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month

    # Original Season Mapping (Linear works better here than cyclical)
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    # Granular Age Bins from your 0.838 version
    df['log_age']      = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # Temperature Anomaly (The "Climate" Signal)
    df['year_avg_temp'] = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']  = df['max_temperature'] - df['year_avg_temp']

    # Year Sensitivity logic
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    # Simple Categorical mapping
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.fillna(-999)

# 3. K-FOLD ENSEMBLE (LGBM + XGB)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()

oof_lgbm, oof_xgb = np.zeros(len(X)), np.zeros(len(X))
test_lgbm, test_xgb = np.zeros(len(X_test)), np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Using 0.838 winning params
    lgbm = LGBMClassifier(n_estimators=2000, learning_rate=0.03, max_depth=6,
                          scale_pos_weight=2.5, colsample_bytree=0.8, subsample=0.8, random_state=42)
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(100), log_evaluation(0)])

    xgb = XGBClassifier(n_estimators=2000, learning_rate=0.03, max_depth=6,
                        scale_pos_weight=neg/pos, colsample_bytree=0.8, subsample=0.8,
                        random_state=42, early_stopping_rounds=100, eval_metric='auc')
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"Fold {fold+1} complete.")

# 4. BLENDING & THRESHOLD OPTIMIZATION
# 60/40 blend often works better if one model (usually LGBM) has a higher AUC
oof_ensemble = (0.6 * oof_lgbm) + (0.4 * oof_xgb)
test_ensemble = (0.6 * test_lgbm) + (0.4 * test_xgb)

best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.3, 0.7, 0.01):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\nFinal OOF Score: {best_score:.4f} at Threshold: {best_thresh:.2f}")

# 5. SUBMISSION
submission = pd.DataFrame({
    'ID': ss['ID'],
    'TargetF1': (test_ensemble >= best_thresh).astype(int),
    'TargetRAUC': test_ensemble
})
submission.to_csv('refined_winning_submission.csv', index=False)
files.download('refined_winning_submission.csv')

In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings('ignore')

# 1. LOAD DATA
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. OPTIMIZED FEATURE ENGINEERING (Retaining Lat/Lon & 6th-Place Logic)
def engineer_features(df):
    df = df.copy()
    df['deathdate'] = pd.to_datetime(df['deathdate'])
    df['year']  = df['deathdate'].dt.year
    df['month'] = df['deathdate'].dt.month
    df['day']   = df['deathdate'].dt.day
    df['dayofweek'] = df['deathdate'].dt.dayofweek

    # Season Mapping
    df['season'] = df['month'].map({
        12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
        6:2, 7:2, 8:2, 9:3, 10:3, 11:3
    })

    # Granular Age Bins
    df['log_age']   = np.log1p(df['age'])
    df['age_squared']  = df['age'] ** 2
    df['is_infant']    = (df['age'] == 0).astype(int)
    df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
    df['is_child']     = (df['age'].between(1, 5)).astype(int)
    df['is_elderly']   = (df['age'] >= 60).astype(int)
    df['age_group']    = pd.cut(df['age'], bins=[-1,0,1,2,5,10,15,30,60,200],
                                labels=[0,1,2,3,4,5,6,7,8]).astype(int)

    # Temperature Anomaly
    df['year_avg_temp'] = df.groupby('year')['max_temperature'].transform('mean')
    df['temp_anomaly']  = df['max_temperature'] - df['year_avg_temp']

    # Year Sensitivity logic (with safe fallback for recent years)
    year_sensitivity = {
        2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
        2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
        2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661, 2023: 0.600, 2024: 0.600,
        2025: 0.600, 2026: 0.600
    }
    df['year_sensitivity'] = df['year'].map(year_sensitivity).fillna(0.6)
    df['year_sens_age']    = df['year_sensitivity'] * df['age']

    # Categorical mapping (keeping lat and lon active)
    df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
    df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

    drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
    return df.drop(drop_cols, axis=1, errors='ignore')

train_df = engineer_features(train)
test_df  = engineer_features(test)

X = train_df.drop('is_climate_sensitive', axis=1)
y = train_df['is_climate_sensitive']
X_test = test_df

# Handle any remaining categorical columns
for c in X.select_dtypes(include=['object', 'category']).columns:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X[c] = oe.fit_transform(X[[c]])
    X_test[c] = oe.transform(X_test[[c]])

X = X.fillna(-999)
X_test = X_test.fillna(-999)

# 3. K-FOLD ENSEMBLE (HistGradientBoosting + RandomForest)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_hgb, oof_rf = np.zeros(len(X)), np.zeros(len(X))
test_hgb, test_rf = np.zeros(len(X_test)), np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Model 1: HistGradientBoosting
    hgb = HistGradientBoostingClassifier(random_state=42, class_weight='balanced', max_iter=200)
    hgb.fit(X_tr, y_tr)

    # Model 2: RandomForest
    rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced', max_depth=10)
    rf.fit(X_tr, y_tr)

    oof_hgb[val_idx] = hgb.predict_proba(X_val)[:, 1]
    oof_rf[val_idx]  = rf.predict_proba(X_val)[:, 1]

    test_hgb += hgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_rf  += rf.predict_proba(X_test)[:, 1] / folds.n_splits
    print(f"Fold {fold+1} complete.")

# 4. BLENDING & THRESHOLD OPTIMIZATION
oof_ensemble = (0.5 * oof_hgb) + (0.5 * oof_rf)
test_ensemble = (0.5 * test_hgb) + (0.5 * test_rf)

best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

final_f1 = f1_score(y, (oof_ensemble >= best_thresh).astype(int))
final_auc = roc_auc_score(y, oof_ensemble)
print(f"\nValidation F1: {final_f1:.4f} | Validation AUC: {final_auc:.4f}")
print(f"Final OOF Score: {best_score:.4f} at Threshold: {best_thresh:.2f}")

# 5. SUBMISSION
submission = pd.DataFrame({
    'ID': ss['ID'],
    'TargetF1': (test_ensemble >= best_thresh).astype(int),
    'TargetRAUC': test_ensemble
})
submission.to_csv('refined_winning_submission_v2.csv', index=False)
print("Submission file saved successfully!")

Fold 1 complete.
Fold 2 complete.
Fold 3 complete.
Fold 4 complete.
Fold 5 complete.

Validation F1: 0.8104 | Validation AUC: 0.8067
Final OOF Score: 0.8089 at Threshold: 0.25
Submission file saved successfully!
